# CEREBRO PoC — 03 Knowledge Construction

**Experiment:** EXP-KNOW-001  
**Stage:** Knowledge Construction  
**Input:** Registered Artifact `ART-0001`

## Objective

Transform a trusted CEREBRO artifact into structured, provenance-aware knowledge fragments.

The same knowledge model will support two interconnected views:

**Knowledge Galaxy** — how knowledge is semantically connected.

**Knowledge Timeline** — when knowledge was created, learned, encountered, or ingested.

Both views resolve to the same underlying knowledge fragments and original source evidence.

### Principle

**One Knowledge Model → Multiple Views**

Registered Artifact → Knowledge Fragments → Semantic + Temporal Relationships → Galaxy / Timeline → Assisted Recollection

### 1 — Load registered artifact

In [1]:
from pathlib import Path
import json

repo_root = Path.cwd().parents[1]

artifact_path = (
    repo_root
    / "poc/data/processed/artifacts/ART-0001.json"
)

assert artifact_path.exists(), (
    f"Registered artifact not found: {artifact_path}"
)

with open(
    artifact_path,
    "r",
    encoding="utf-8"
) as f:
    artifact = json.load(f)

assert artifact["status"] == "registered"

print("✓ Registered artifact loaded")
print("Artifact :", artifact["artifact_id"])
print("Source   :", artifact["source"]["filename"])

✓ Registered artifact loaded
Artifact : ART-0001
Source   : benchmark_001.txt


## 2. Resolve Source Evidence

Knowledge construction must operate against the registered source artifact.

CEREBRO verifies that the source used for knowledge extraction is the same source that was registered during artifact intake.

This preserves the integrity chain:

**Knowledge Fragment → Registered Artifact → Original Source**

### 2 — Verify source

In [2]:
import hashlib

source_path = (
    repo_root
    / artifact["source"]["storage_uri"]
)

assert source_path.exists()

source_bytes = source_path.read_bytes()

source_sha256 = hashlib.sha256(
    source_bytes
).hexdigest()

assert (
    source_sha256
    == artifact["source"]["sha256"]
)

source_text = source_bytes.decode("utf-8")

print("✓ Original source resolved")
print("✓ SHA-256 verified")
print()
print(source_text)

✓ Original source resolved
✓ SHA-256 verified

CEREBRO is a Digital Knowledge Twin designed to preserve and connect human knowledge. It maintains provenance between knowledge fragments and their original source artifacts. The system supports assisted recollection by allowing users to navigate from knowledge back to its supporting evidence.


### 3 — Sentence segmentation

In [3]:
import re

sentences = [
    sentence.strip()
    for sentence in re.split(
        r'(?<=[.!?])\s+',
        source_text
    )
    if sentence.strip()
]

print(
    f"✓ {len(sentences)} candidate fragments created\n"
)

for i, sentence in enumerate(sentences, start=1):
    print(f"{i}. {sentence}")

✓ 3 candidate fragments created

1. CEREBRO is a Digital Knowledge Twin designed to preserve and connect human knowledge.
2. It maintains provenance between knowledge fragments and their original source artifacts.
3. The system supports assisted recollection by allowing users to navigate from knowledge back to its supporting evidence.


### 4 — Create canonical fragments

In [4]:
knowledge_fragments = []

search_start = 0

for index, sentence in enumerate(
    sentences,
    start=1
):

    start_char = source_text.find(
        sentence,
        search_start
    )

    end_char = start_char + len(sentence)

    fragment = {
        "fragment_id": f"KF-{index:04d}",

        "artifact_id": artifact["artifact_id"],

        "content": sentence,

        "source_location": {
            "type": "character_range",
            "start": start_char,
            "end": end_char
        },

        "semantic": {
            "concepts": [],
            "entities": [],
            "relationships": []
        },

        "temporal": {
            "source_date": None,
            "knowledge_date": None,
            "ingested_at": None
        },

        "provenance": {
            "source_artifact": artifact["artifact_id"],
            "source_filename": artifact[
                "source"
            ]["filename"],
            "source_sha256": artifact[
                "source"
            ]["sha256"],
            "construction_method":
                "deterministic_sentence_segmentation",
            "experiment_id": "EXP-KNOW-001"
        }
    }

    knowledge_fragments.append(fragment)

    search_start = end_char

print(
    f"✓ {len(knowledge_fragments)} "
    "knowledge fragments constructed"
)

✓ 3 knowledge fragments constructed


## 4. Verify Fragment Provenance

CEREBRO must be able to reconstruct the exact source evidence supporting every knowledge fragment.

This validates:

**CIF-01 — No derived knowledge without provenance.**

**CIF-02 — Trusted knowledge must resolve back to its original source and source location.**

### 5 — Reconstruct every fragment

In [5]:
for fragment in knowledge_fragments:

    location = fragment["source_location"]

    reconstructed = source_text[
        location["start"]:
        location["end"]
    ]

    assert reconstructed == fragment["content"]

    print(
        f'✓ {fragment["fragment_id"]} '
        "reconstructed exactly"
    )

✓ KF-0001 reconstructed exactly
✓ KF-0002 reconstructed exactly
✓ KF-0003 reconstructed exactly


## 5. Semantic Enrichment

Knowledge fragments require semantic structure before they can participate in the Knowledge Galaxy.

For this controlled PoC benchmark, concepts and relationships are explicitly defined so that the semantic model can be validated before automated extraction is introduced.

Automated entity and relationship extraction can later replace this controlled baseline.

### 6 — Add concepts

In [6]:
semantic_annotations = {
    "KF-0001": {
        "concepts": [
            "CEREBRO",
            "Digital Knowledge Twin",
            "Human Knowledge"
        ]
    },

    "KF-0002": {
        "concepts": [
            "Provenance",
            "Knowledge Fragment",
            "Source Artifact"
        ]
    },

    "KF-0003": {
        "concepts": [
            "Assisted Recollection",
            "Knowledge Navigation",
            "Supporting Evidence"
        ]
    }
}

for fragment in knowledge_fragments:

    fragment_id = fragment["fragment_id"]

    annotation = semantic_annotations.get(
        fragment_id,
        {"concepts": []}
    )

    fragment["semantic"]["concepts"] = (
        annotation["concepts"]
    )

print("✓ Semantic concepts attached")

✓ Semantic concepts attached


### 7 — Inspect

In [7]:
for fragment in knowledge_fragments:

    print(
        "\n",
        fragment["fragment_id"],
        "—",
        fragment["content"]
    )

    print(
        "Concepts:",
        ", ".join(
            fragment["semantic"]["concepts"]
        )
    )


 KF-0001 — CEREBRO is a Digital Knowledge Twin designed to preserve and connect human knowledge.
Concepts: CEREBRO, Digital Knowledge Twin, Human Knowledge

 KF-0002 — It maintains provenance between knowledge fragments and their original source artifacts.
Concepts: Provenance, Knowledge Fragment, Source Artifact

 KF-0003 — The system supports assisted recollection by allowing users to navigate from knowledge back to its supporting evidence.
Concepts: Assisted Recollection, Knowledge Navigation, Supporting Evidence


## 6. Temporal Model

CEREBRO distinguishes different meanings of time.

A source may contain an explicit historical date, while the artifact itself may have a creation date, upload date, or ingestion date.

These values must not be treated as equivalent.

For the current benchmark, no reliable historical knowledge date is explicitly present in the source. CEREBRO therefore leaves unknown temporal values empty rather than inventing them.

This allows the same knowledge fragments to later participate safely in the Knowledge Timeline.

### 8 — Temporal classification

In [8]:
from datetime import datetime, timezone

ingestion_time = datetime.now(
    timezone.utc
).isoformat()

for fragment in knowledge_fragments:

    fragment["temporal"] = {
        "source_date": None,
        "knowledge_date": None,
        "ingested_at": ingestion_time,
        "temporal_confidence": {
            "source_date": "unknown",
            "knowledge_date": "unknown",
            "ingested_at": "verified"
        }
    }

print("✓ Temporal metadata attached")
print("Ingested:", ingestion_time)

✓ Temporal metadata attached
Ingested: 2026-09-24T05:18:04.885262+00:00


## 7. Knowledge Relationships

Knowledge relationships connect fragments without duplicating them.

These relationships will later drive the Knowledge Galaxy.

Temporal metadata attached to the same fragments will drive the Knowledge Timeline.

Selecting a fragment in either view therefore refers to the same underlying knowledge object.

### 9 — Controlled relationships

In [9]:
relationships = [
    {
        "relationship_id": "REL-0001",
        "source": "KF-0001",
        "target": "KF-0002",
        "type": "SUPPORTED_BY",
        "reason": (
            "CEREBRO's Digital Knowledge Twin "
            "maintains provenance between knowledge "
            "and source artifacts."
        )
    },

    {
        "relationship_id": "REL-0002",
        "source": "KF-0002",
        "target": "KF-0003",
        "type": "ENABLES",
        "reason": (
            "Provenance enables recollection to "
            "resolve knowledge back to supporting evidence."
        )
    },

    {
        "relationship_id": "REL-0003",
        "source": "KF-0001",
        "target": "KF-0003",
        "type": "SUPPORTS",
        "reason": (
            "The Digital Knowledge Twin supports "
            "assisted recollection."
        )
    }
]

print(
    f"✓ {len(relationships)} "
    "knowledge relationships created"
)

✓ 3 knowledge relationships created


### 10 - Project Models

In [10]:
knowledge_model = {
    "model_id": "KM-0001",

    "artifact_id": artifact["artifact_id"],

    "fragments": knowledge_fragments,

    "relationships": relationships,

    "projections": {
        "galaxy": {
            "basis": "semantic_relationships"
        },
        "timeline": {
            "basis": "temporal_metadata"
        }
    }
}

print("✓ Unified knowledge model created")
print()
print("Fragments    :", len(knowledge_fragments))
print("Relationships:", len(relationships))
print("Views        : Galaxy + Timeline")

✓ Unified knowledge model created

Fragments    : 3
Relationships: 3
Views        : Galaxy + Timeline


## 8. Persist Knowledge Model

The resulting model becomes the shared source for downstream CEREBRO capabilities.

The Knowledge Galaxy and Knowledge Timeline must not maintain independent copies of knowledge.

Both views will consume this persisted model.

### 11 - Persist model

In [11]:
output_dir = (
    repo_root
    / "poc/data/processed/knowledge"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    output_dir
    / "KM-0001.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        knowledge_model,
        f,
        indent=2,
        ensure_ascii=False
    )

print("✓ Knowledge model persisted")
print("Output:", output_path)

✓ Knowledge model persisted
Output: /Users/joeldizon/development/cerebro_dev/cerebro/poc/data/processed/knowledge/KM-0001.json


### 12 - Integrity test

In [12]:
assert len(
    knowledge_model["fragments"]
) > 0

assert len(
    knowledge_model["relationships"]
) > 0

for fragment in knowledge_model["fragments"]:

    location = fragment["source_location"]

    evidence = source_text[
        location["start"]:
        location["end"]
    ]

    assert evidence == fragment["content"]

print("✓ Knowledge fragments created")
print("✓ Semantic concepts attached")
print("✓ Temporal metadata attached")
print("✓ Relationships constructed")
print("✓ Exact source evidence verified")
print("✓ Galaxy projection supported")
print("✓ Timeline projection supported")

print("\nEXP-KNOW-001: PASS")

✓ Knowledge fragments created
✓ Semantic concepts attached
✓ Temporal metadata attached
✓ Relationships constructed
✓ Exact source evidence verified
✓ Galaxy projection supported
✓ Timeline projection supported

EXP-KNOW-001: PASS


## Experiment Result

CEREBRO successfully transformed a registered artifact into a unified semantic and temporal knowledge model.

The experiment demonstrates:

**ART-0001 → Knowledge Fragments → Concepts → Relationships → Semantic + Temporal Model**

The same knowledge fragments can now support:

**Knowledge Galaxy**  
Semantic exploration of connected knowledge.

**Knowledge Timeline**  
Temporal exploration of the user's knowledge journey.

**Assisted Recollection**  
Navigation from knowledge back to exact supporting evidence.

No separate Galaxy or Timeline knowledge stores are required.

### Next Experiment

**EXP-KNOW-002 — Embeddings & Semantic Relationship Discovery**

The next experiment will use vector embeddings to determine whether CEREBRO can discover relationships between knowledge fragments automatically rather than relying only on manually defined relationships.